# 005 · Volatility model

**The question.** How much will this stock's price swing over the next three months?

**The answer.** `vol_63_future`: the annualised volatility of daily returns over the next 63
trading days (risk_params v5), modelled as its logarithm. The forecast is shown as **a level, an
80% range and one of five bands** with the band's past outcomes (D62, D47), never as one bare
number.

**The baseline to beat** (pre-registered): `vol_63`, the same measure over the past 63 days,
assumed to continue.

**How the levels are judged** (V2-56): the mean absolute error of the log; the root mean squared
error of the log, rank correlation and the 80% range's coverage beside it. The simplest level
within one standard error of the yearly gaps to the best is chosen (V2-50).

**Status.** Built on 9 November 2026 ([spec](../specs/005-volatility-model/spec.md),
[plan](../specs/005-volatility-model/plan.md),
[research](../specs/005-volatility-model/research.md), issue #67). Built only if approved by the
evening of 11 November (V2-48). Gemini's first version (30 Oct – 3 Nov) was not approved: no
one-standard-error rule, a range scaled on its own training rows (65–67% coverage), bands without
outcomes (audit, 4 Nov). **After the first full run the macro family was left out of the ridge and
LightGBM (V2-57, 9 Nov)**: decided after seeing the test years, so the comparison below flatters the
models a little; the holdout test (010) is the honest check.

### What the answer means (read this first)

Think of an exam: the **features** are the questions (what was known about a company at a month
end: how much its price has moved lately, its size, its finances, recent filings, the market's
mood), and **`vol_63_future`** is the answer key (how much the price really swung over the next
three months). The model learns from rows *with* the key and is judged on rows whose key it never
saw.

- **The answer, with an example.** Volatility is the typical size of a day's price change, scaled
  to a year. A stock with a volatility of 0.20 (20% a year) moves about 1.3% on a typical day; one
  with 0.80 about 5%. If a stock's daily moves over the next 63 trading days have a spread of
  2.5%, its answer is 2.5% × √252 ≈ 0.40.
- **Why the logarithm.** Volatilities run from 0.1 to above 2, and a miss of 0.1 means much more for
  a calm stock than for a wild one. On the log scale a miss of 0.30 means about 35% too high or too
  low, for any stock. The deciding measure is the average size of these log misses (V2-56): the
  naive baseline scores about 0.25–0.49 in the test years.
- **The level, the range, the band.** The level is the middle expectation (half the time the stock
  swings more, half the time less). The 80% range should hold the answer eight times in ten; its
  width comes from how far forecasts missed in all earlier years (V2-10), never from the training
  rows' own errors, which are too small. The five bands group stocks by their forecast, each with
  what really happened to stocks in it.
- **Which rows and columns.** Fit only on `volatility.parquet`, rows with `eligible_volatility`
  true and an answer above 0 (the 93 answers of exactly 0 have no logarithm); the inputs are the
  features known at the month-end close: 79 for the ridge and LightGBM, without the macro family
  (V2-57).
- **How error is measured, in three stages, always on answers the model has not seen:** (1) the
  walk-forward test years 2019–2022 here; (2) the holdout, once (feature 010); (3) live (feature
  009, after 010): each day's forecast is stored and scored three months later.

### Rules for every model notebook

- **Spec first.** The spec, plan and tasks are in `specs/005-volatility-model/`, on branch
  `005-volatility-model`.
- **This notebook is where the model is built.** Every level is created, run and tuned here,
  start to finish by one command, so the researcher can read how the choice was made. Once the
  researcher approves the chosen model, its code moves into `src/sentinel/models/volatility.py`
  and this notebook imports it and must give the same numbers.
- **Data.** Read only `data/exports/ml/volatility.parquet` through `sentinel.eda.load("volatility")`
  (rows dated 2022 or earlier). Never read `data/exports/reference/` and never compute anything
  for 2023 or later: that is the sealed holdout, tested once by feature 010 (constitution IV).
- **Never use as inputs:** `vol_63_future`, `window_end`, `cik`, `as_of`, the flags, `sector` (each
  company's *current* industry code, not point-in-time) and `cum_split_after` (look-ahead). **The
  ridge and LightGBM leave out the macro family** (V2-57, decided after the first full run: it made
  LightGBM's 2022 error 0.138 worse when rates rose faster than in any training year, and added
  nothing on 2016–2018); H-V4 is checked by adding it back.
- **Never a random split.** The only split is walk-forward by date (`eda.folds`, checked by
  `eda.check_folds`).
- **Evaluation is fixed** (D60): test years 2019–2022; four levels: naive baseline → rule →
  classic model → LightGBM; the **simplest level within one standard error of the best** (V2-50:
  the standard error of the four yearly differences); every test year reported.
- **Every fitted step knows when it learned (V2-49):** each forecast, range and set of band edges
  is a `Dated`: it learns only from rows whose answers were known before its cutoff and refuses
  rows dated before it. No early stopping.
- **The method's choices were made before any test year** (research.md, design check on
  2014–2018): the ridge and LightGBM forecast the *change* from past volatility; the range is set on
  all earlier years' out-of-sample errors. To give it that history the walk-forward runs from 2014;
  only 2019–2022 are scored.
- **Reproducible:** rows sorted by `cik`, `as_of`; fixed seeds (`SEED = 0`); each run logged in
  MLflow (experiment `005-volatility`) with the git commit, the export date and a data hash.
- **Private data:** outputs are stripped on commit (nbstripout); written conclusions quote
  aggregates only.
- **Run it** start to finish under the memory cap:
  `systemd-run --user --scope -p MemoryMax=5G -p MemorySwapMax=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/005-volatility-model.ipynb`
  (with `SENTINEL_TRIAL=1` it runs one setting and one seed: a check of the code, not results).

### Two kinds of test, and the final one is not in this notebook

1. **Test years 2019–2022 (here).** For each test year, learn from the earlier years only and
   score that year. This is where the four levels are compared and **one is chosen**. Because the
   choice is made by looking at these results, they flatter the winner a little.
2. **The holdout: answers whose windows start in 2023 or later (not here, not in any notebook).**
   Does the model chosen here work on data nobody looked at while choosing it? That works only if
   it is used **once**: one guarded command (feature 010, 16 November) computes the 2023+ answers,
   scores the frozen models and writes the results with a log entry; a second run is refused.

**Holdout-ready:** inputs selected by name; every preprocessing step, the range and the band edges
inside the model; one saved object; a dry run on the 2023+ month-end rows of
`feature_snapshot.parquet` and on `latest.parquet` (features only, no answers) before 16 November;
scoring by one function `predict(model, rows)`.

### What the EDA found for this model (notebooks/004-eda-volatility.ipynb, Findings)

- 531,000 rows (7,900 companies) eligible with an answer, 2012–2022; 9,500 of price-only stocks
  (V2-26), which trade more thinly: report them separately.
- Median answer 0.36 (±36% a year); 80% of rows between 0.18 and 0.83; small companies swing more
  (micro 0.51, large 0.25); 2020 stands out.
- **The naive baseline** ranks well (rank correlation 0.79) but misses by about 30% on a typical
  row (mean absolute log error 0.35, 0.49, 0.29, 0.25 in 2019–2022), badly in 2020.
- **Mean reversion**: log(future) = −0.255 + 0.776 × log(past): volatility moves about a fifth of
  the way back toward normal each quarter. This rule beats the baseline in 2019–2021.
- Strongest features: past volatility in its forms, the fall from the 12-month high, price level,
  loss-making, interest coverage, size, R&D intensity, cash share; for the *change*: market
  volatility, VIX and 2-year rate changes, unemployment. Weak: insider activity, short volume.
- Redundant pairs: one number on two scales (`close`/`log_close` and the like), insider counts and
  values, `vol_252`/`idio_vol_252`, `vol_21`/`range_vol_21`.

**Data cleaning** (V2-33): the pipeline's cleaning is frozen; further cleaning (logs, cuts at the
1st and 99th percentiles, "was missing" flags) is part of this model, fitted on training rows only.

## 1. Setup and data

The development rows (month ends, January 2012 – September 2022) with this question's flag and an
answer above 0; the 95 features, of which the ridge and LightGBM use the 79 outside the macro
family (V2-57); each row also gets `log_vol` (the log of its answer) and whether it is a
price-only stock.

In [ ]:
# Setup: libraries, the rows the model learns from, the features, MLflow
import os
import re
import tempfile
import warnings
from dataclasses import dataclass, field
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")  # keep MLflow's tips out of the outputs

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

from sentinel import config, eda
from sentinel.models import dilution  # the approved 007 code: Bands, provenance (V2-52)
from sentinel.models.dated import Dated

# SENTINEL_TRIAL=1 runs the code quickly (one setting, one seed): a check of the code, not results
TRIAL = os.environ.get("SENTINEL_TRIAL") == "1"
SEED, THREADS = 0, 8  # LightGBM on 8 of the 16 cores
threadpool_limits(THREADS)
warnings.filterwarnings("ignore", category=RuntimeWarning)  # nan-quantiles of empty columns
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
P = config.paths()

QUESTION, ANSWER, ELIG = "volatility", "vol_63_future", "eligible_volatility"
FLOOR = 1e-4  # the log of past volatility uses at least this (a price that never moved)

raw = eda.load(QUESTION)
flag = raw[ELIG].fillna(False).astype(bool)
zero = int((flag & (raw[ANSWER] == 0)).sum())
rows = raw[flag & (raw[ANSWER] > 0)]  # the answers of exactly 0 have no logarithm
rows = rows.sort_values(["cik", "as_of"], kind="stable").reset_index(drop=True)

# Inputs: the export's features. Never the answer, its window end, identifiers, flags, the sector
# (current SIC, not point-in-time) or cum_split_after (look-ahead)
FEATURES = [c for c in eda.features(rows) if c not in ("cum_split_after", "sector")]
NEVER = {"vol_63_future", "window_end", "cik", "as_of", "sector", "cum_split_after", "month_end",
         "release_eve", "latest", "coverage", "filer_type", "foreign_filer", "active_filer",
         "excluded", "eligible", "eligible_volatility", "eligible_dilution"}
assert not NEVER & set(FEATURES) and len(FEATURES) == 95, len(FEATURES)
FAMILY = {c: eda.family(c) for c in FEATURES}
# The macro family is left out of the ridge and LightGBM (V2-57, after the first full run); H-V4
# adds it back in section 9b
MACRO = [c for c in FEATURES if FAMILY[c] == "macro"]
MODEL_FEATURES = [c for c in FEATURES if c not in MACRO]
rows = rows.astype({c: float for c in FEATURES})
derived = {"log_vol": np.log(rows[ANSWER]),
           "log_past": np.log(np.maximum(rows["vol_63"], FLOOR)),
           "price_only": rows["coverage"].eq("price_only")}
rows = pd.concat([rows, pd.DataFrame(derived, index=rows.index)], axis=1).copy()  # one block

print(f"{len(rows):,} rows, {rows.cik.nunique():,} companies ({zero} answers of 0 left out); "
      f"{int(rows.price_only.sum()):,} rows of price-only stocks; {len(FEATURES)} features, "
      f"{len(MODEL_FEATURES)} without the {len(MACRO)} macro features (V2-57)")
print(f"answer: median {rows[ANSWER].median():.3f}, 80% of rows between "
      f"{rows[ANSWER].quantile(0.1):.3f} and {rows[ANSWER].quantile(0.9):.3f}")
display(rows.assign(year=rows.as_of.dt.year).groupby("year").agg(
    rows=("cik", "size"), companies=("cik", "nunique"), median_vol=(ANSWER, "median"),
    price_only=("price_only", "sum")))

# MLflow: runs logged to data/mlflow.db, artifacts to data/mlartifacts (research R10)
mlflow.set_tracking_uri(f"sqlite:///{P.mlflow_db}")
EXPERIMENT = "005-volatility" + ("-trial" if TRIAL else "")  # trial runs kept apart
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT,
                             artifact_location=(P.mlflow_db.parent / "mlartifacts").as_uri())
mlflow.set_experiment(EXPERIMENT)
SRC = eda.EXPORT / "volatility.parquet"
TAGS = {**dilution.provenance(SRC), "level3_estimator": f"lightgbm {lgb.__version__}",
        "rules": "Dated (V2-49); one SE of the yearly gaps (V2-50); range on all earlier years "
                 "(V2-10); mean absolute log error (V2-56); no macro in the models (V2-57)",
        "trial": str(TRIAL)}
print(TAGS)

## 2. Walk-forward folds

For each year Y the rows fall in blocks by their date and the day their answer was known
(`window_end`, three months later; `eda.folds`): **inner** (answer known before Y−1: the forecasts
learn from these), **gap** (dated before Y−1, known during Y−1: left out), **valid** (dated in Y−1,
known before Y: settings), **purged** (known in Y or later: left out) and **test** (dated in Y).
The walk-forward runs from **2014**: the years 2014–2018 are forecast exactly like the test years
but **not scored**; their out-of-sample errors give every later year's 80% range and band edges
their history (V2-10). Only **2019–2022** are scored.

The helpers below are shared by every level. Each level forecasts the log volatility of the test
year; its errors join the level's history; for a scored year the range is the 10th and 90th
percentiles of the history's errors whose answers were known before the year, and the band edges
are the quintiles of the history's forecasts (both `Dated`, cutoff 1 January of the year).

In [ ]:
# Walk-forward folds cut by dates and checked (eda.folds, eda.check_folds), and the helpers
YEARS = list(range(2014, eda.DEV_END.year + 1))  # 2014-2018: history for the range; then scored
SCORED = list(eda.TEST_YEARS)  # 2019-2022
FOLDS = eda.folds(rows, QUESTION, YEARS)
display(eda.check_folds(rows, FOLDS, QUESTION))  # refuses any block out of time order
for f in FOLDS.values():
    f["train"] = f["inner"] | f["gap"] | f["valid"]  # every answer known before the year
fold_table = pd.DataFrame({year: {k: int(m.sum()) for k, m in f.items()}
                           for year, f in FOLDS.items()}).T
display(fold_table)
expected = eda.purge_table(rows, "window_end", YEARS)
assert (fold_table[["train", "purged", "test"]].values
        == expected[["train_rows", "purged", "test_rows"]].values).all()
print("fold counts equal the EDA's purge table")


def valid_start(year: int) -> pd.Timestamp:
    """The cutoff of the forecasts judged on year `year`: answers known before its validation year."""
    return pd.Timestamp(year - 1, 1, 1)


def test_start(year: int) -> pd.Timestamp:
    """The cutoff of the settings, the range and the band edges used on year `year`."""
    return pd.Timestamp(year, 1, 1)


def mae_log(forecast, mask) -> float:
    """The mean absolute error of the log (V2-56): how settings and levels are chosen."""
    return float(np.mean(np.abs(np.asarray(forecast) - rows.loc[mask, "log_vol"].to_numpy())))


def past_of(X) -> np.ndarray:
    """log(vol_63): the naive baseline, and the starting point of the change models."""
    return np.log(np.maximum(X["vol_63"].to_numpy(dtype=float), FLOOR))


class Change(BaseEstimator, RegressorMixin):
    """A model of the change from past volatility (research R3): it learns log(future) − log(past)
    on `cols` and forecasts log(past) + the change, so it starts from the naive baseline whatever
    its inputs."""

    def __init__(self, base, cols):
        self.base, self.cols = base, cols

    def fit(self, X, y):
        self.base.fit(X[self.cols], np.asarray(y, dtype=float) - past_of(X))
        return self

    def predict(self, X):
        return past_of(X) + self.base.predict(X[self.cols])


def with_past(cols) -> list:
    """The columns a change model reads: its inputs and `vol_63` for its starting point."""
    return list(cols) + ([] if "vol_63" in cols else ["vol_63"])


class Range(BaseEstimator):
    """The 80% range (V2-10): the 10th and 90th percentiles of earlier years' out-of-sample errors
    (forecast minus answer, in logs). Around a forecast f the answer lies between f − q90 and
    f − q10 eight times in ten, if the future errs like the past."""

    def fit(self, X, y):
        self.q10_, self.q90_ = (float(v) for v in np.quantile(np.asarray(y, float), [0.1, 0.9]))
        return self

    def predict(self, X):
        f = np.asarray(X, dtype=float)[:, 0]
        return np.column_stack([f - self.q90_, f - self.q10_])


def fit_level(make, f, year, grid) -> dict:
    """Learn from the inner rows (answers known before the validation year); choose the setting by
    the validation year's mean absolute log error (research R3). `make(**setting)` gives the
    unfitted `Dated`."""
    tried = {}
    for params in grid:
        model = make(**params)
        model.fit(rows[f["inner"]], valid_start(year), y=rows.loc[f["inner"], "log_vol"])
        tried[str(params)] = (mae_log(model.predict(rows[f["valid"]]), f["valid"]), params, model)
    _, params, model = min(tried.values(), key=lambda t: t[0])
    return {"model": model, "params": params, "setting": params,
            "tried": {k: round(v[0], 5) for k, v in tried.items()}}


HIST, FITTED, RESULTS, TEST = {}, {}, [], {}


def earlier(level: str, year: int) -> pd.DataFrame:
    """The level's out-of-sample history before `year`: the forecasts and errors of every earlier
    year whose answers were known before `year`."""
    h = pd.concat([t for t in HIST[level] if t["year"].iloc[0] < year])
    return h[h["window_end"] < test_start(year)]


def scores(mask, forecast, low, high) -> dict:
    """The mean absolute log error (deciding, V2-56), the root mean squared log error, the rank
    correlation with the answer and the 80% range's coverage; for price-only stocks and US filers
    apart."""
    y = rows.loc[mask, "log_vol"].to_numpy()
    err, inside = forecast - y, (y >= low) & (y <= high)
    po = rows.loc[mask, "price_only"].to_numpy()
    return {"mae_log": float(np.mean(np.abs(err))), "rmse_log": float(np.sqrt(np.mean(err ** 2))),
            "rank_corr": float(pd.Series(forecast).corr(pd.Series(y), method="spearman")),
            "coverage_80": float(inside.mean()),
            "mae_log_filers": float(np.mean(np.abs(err[~po]))),
            "mae_log_price_only": float(np.mean(np.abs(err[po]))) if po.any() else np.nan,
            "coverage_price_only": float(inside[po].mean()) if po.any() else np.nan,
            "rows": int(len(y)), "rows_price_only": int(po.sum())}


def run(level: str, year: int, fit: dict) -> dict | None:
    """Forecast the year's test rows; keep their errors in the level's history; for a scored year,
    set the range and the band edges on the history before it, score and log one MLflow run."""
    f = FOLDS[year]
    test = rows[f["test"]]
    forecast = fit["model"].predict(test)
    HIST.setdefault(level, []).append(pd.DataFrame(
        {"as_of": test["as_of"], "window_end": test["window_end"], "forecast": forecast,
         "error": forecast - test["log_vol"].to_numpy(), "year": year}, index=test.index))
    FITTED[(level, year)] = fit
    if year not in SCORED:
        return None
    hist = earlier(level, year)
    rng = Dated(Range(), QUESTION, ["forecast"]).fit(hist, test_start(year), y=hist["error"])
    bands = Dated(dilution.Bands(), QUESTION, ["forecast"]).fit(hist, test_start(year),
                                                                 y=hist["error"])
    frame = test[["as_of"]].assign(forecast=forecast)
    low, high = rng.predict(frame).T
    m = scores(f["test"], forecast, low, high)
    TEST[(level, year)] = {"forecast": forecast, "low": low, "high": high,
                           "band": bands.predict(frame), "range": rng, "bands": bands,
                           "history_years": sorted(hist["year"].unique().tolist())}
    RESULTS.append({"level": level, "year": year, **m})
    with mlflow.start_run(run_name=f"{level}-{year}"):
        mlflow.set_tags({**TAGS, "level": level, "test_year": year})
        mlflow.log_params(fit["params"])
        mlflow.log_metrics({k: v for k, v in m.items() if np.isfinite(v)})
    return m

## 3. Level 0: the naive baseline

Past volatility continues: the forecast is `log(vol_63)`, the volatility of the past 63 sessions.
Nothing is learned; its range still comes from how far it missed in earlier years.

In [ ]:
# Level 0: the pre-registered naive baseline, past volatility continues
class Baseline(BaseEstimator, RegressorMixin):
    """log(vol_63): the past three months' volatility, assumed to continue."""

    def fit(self, X, y):
        return self

    def predict(self, X):
        return past_of(X)


for year, f in FOLDS.items():
    run("baseline", year, fit_level(lambda: Dated(Baseline(), QUESTION, ["vol_63"]), f, year,
                                    [{}]))
display(pd.DataFrame(RESULTS).query("level == 'baseline'").set_index("year"))

## 4. Level 1: a rule

The EDA's mean-reversion line: **log(future) = a + b · log(past)**, with `a` and `b` fitted by
least squares on the inner rows. A slope `b` below 1 means volatility drifts back toward normal:
a wild stock calms somewhat, a calm one stirs (hypothesis H-V1).

In [ ]:
# Level 1: the mean-reversion rule log(future) = a + b * log(past)
class Rule(BaseEstimator, RegressorMixin):
    """log(future) = a + b * log(vol_63), by least squares."""

    def fit(self, X, y):
        self.lr_ = LinearRegression().fit(past_of(X).reshape(-1, 1), y)
        return self

    def predict(self, X):
        return self.lr_.predict(past_of(X).reshape(-1, 1))


lines = {}
for year, f in FOLDS.items():
    fit = fit_level(lambda: Dated(Rule(), QUESTION, ["vol_63"]), f, year, [{}])
    run("rule", year, fit)
    lr = fit["model"].estimator.lr_
    lines[year] = {"a": lr.intercept_, "b (slope)": lr.coef_[0]}
display(pd.DataFrame(lines).T)
SLOPES = pd.DataFrame(lines).T["b (slope)"]
display(pd.DataFrame(RESULTS).query("level == 'rule'").set_index("year"))

## 5. Level 2: a classic model

A ridge regression of the **change** from past volatility (research R3), so its penalty pulls it
toward the naive baseline, on the 79 features without the macro family (V2-57), with the
model-side cleaning of V2-33 fitted on the inner rows only: the
volatility measures as logs; one of each pair of near-duplicate columns (|Spearman| ≥ 0.9 on the
2019 fold's inner rows, fixed for every fold); every column cut at its 1st and 99th percentiles;
missing values filled with the median plus a "was missing" indicator; standardised. The penalty's
strength is chosen on the validation year.

In [ ]:
# Level 2: a ridge regression of the change from past volatility, with the cleaning of V2-33
VOL = ["vol_21", "vol_63", "vol_252", "range_vol_21", "idio_vol_252"]


class Logs(BaseEstimator, TransformerMixin):
    """The log (at least FLOOR) of the columns `cols` among `columns`; the others unchanged."""

    def __init__(self, columns, cols):
        self.columns, self.cols = columns, cols

    def fit(self, X, y=None):
        self.idx_ = [self.columns.index(c) for c in self.cols if c in self.columns]
        return self

    def transform(self, X):
        X = np.array(X, dtype=float)
        X[:, self.idx_] = np.log(np.maximum(X[:, self.idx_], FLOOR))
        return X


class Clip(BaseEstimator, TransformerMixin):
    """Cut each column at its 1st and 99th percentiles of the rows it was fitted on."""

    def __init__(self, lower=0.01, upper=0.99):
        self.lower, self.upper = lower, upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lo_ = np.nanquantile(X, self.lower, axis=0)
        self.hi_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lo_, self.hi_)


# One of each |Spearman| >= 0.9 pair, found on the 2019 fold's inner rows (features against
# features, no answer used; fixed for every fold), keeping the log versions and key features
KEEP = {"log_addv20", "log_market_cap", "log_close", "idio_vol_252", "vol_63"}
pairs = eda.redundant_pairs(rows[FOLDS[2019]["inner"]], MODEL_FEATURES, threshold=0.9, seed=SEED)
redundant = []
for a, b in zip(pairs.a, pairs.b, strict=True):
    if a in redundant or b in redundant or (a in KEEP and b in KEEP):
        continue
    redundant.append(a if b in KEEP else b)
LINEAR = [c for c in MODEL_FEATURES if c not in redundant]
print(f"{len(LINEAR)} columns; left out as redundant: {redundant}")


def ridge(alpha: float, cols=None) -> Dated:
    """The classic model on `cols` (default: LINEAR), as a change from past volatility."""
    cols = list(cols or LINEAR)
    pipe = make_pipeline(Logs(cols, VOL), Clip(), SimpleImputer(strategy="median",
                         add_indicator=True), StandardScaler(), Ridge(alpha=alpha))  # fmt: skip
    return Dated(Change(pipe, cols), QUESTION, with_past(cols))


ALPHAS = [100.0] if TRIAL else [1.0, 10.0, 100.0, 1000.0]
for year, f in FOLDS.items():
    fit = fit_level(ridge, f, year, [{"alpha": a} for a in ALPHAS])
    run("ridge", year, fit)
    if year in SCORED:
        print(year, "validation error by alpha:", fit["tried"], "-> alpha", fit["params"]["alpha"])
display(pd.DataFrame(RESULTS).query("level == 'ridge'").set_index("year"))

## 6. Level 3: LightGBM

Gradient-boosted trees (D60) forecasting the **change** from past volatility (research R3) from
the 79 features without the macro family (V2-57), with squared-error loss and fixed seeds; missing
values are left to the trees.
They learn from the inner rows; the settings (leaves, rows per leaf) and the number of trees are
chosen by the validation year's mean absolute log error, never by early stopping inside the fit
(V2-49).

In [ ]:
# Level 3: LightGBM's forecast of the change from past volatility (D60); its settings and number
# of trees chosen on the validation year (no early stopping, V2-49)
TREES = [50, 100, 150, 200, 300, 400, 600, 800, 1000]
GRID = [{"num_leaves": n, "min_child_samples": m} for n in (15, 63) for m in (100, 500)]
if TRIAL:
    TREES, GRID = [100, 200], GRID[:1]


class BoostRegressor(BaseEstimator, RegressorMixin):
    """LightGBM forecasting with its first `trees_` trees: the number of trees is a setting chosen
    by predicting the validation year, never by early stopping inside the fit (V2-49)."""

    def __init__(self, params=None, n_estimators=1000, seed=SEED):
        self.params, self.n_estimators, self.seed = params, n_estimators, seed

    def fit(self, X, y):
        self.model_ = lgb.LGBMRegressor(
            objective="regression", learning_rate=0.05, n_estimators=self.n_estimators,
            colsample_bytree=0.8, subsample=0.8, subsample_freq=1, deterministic=True,
            force_row_wise=True, n_jobs=THREADS, random_state=self.seed, verbose=-1,
            **(self.params or {})).fit(X, y)
        self.trees_ = self.n_estimators
        return self

    def predict(self, X):
        return self.model_.predict(X, num_iteration=self.trees_)


def fit_boost(f, year, cols=None, seed=SEED, setting=None) -> dict:
    """LightGBM for one fold: one fit of the most trees per setting; the setting and the number of
    trees chosen by the validation year's mean absolute log error. With `setting`, only that one."""
    cols = list(cols or MODEL_FEATURES)
    tried, fitted = {}, {}
    for params in [setting["params"]] if setting else GRID:
        n = setting["trees"] if setting else max(TREES)
        model = Dated(Change(BoostRegressor(params, n, seed), cols), QUESTION, with_past(cols))
        model.fit(rows[f["inner"]], valid_start(year), y=rows.loc[f["inner"], "log_vol"])
        fitted[str(params)] = model
        for k in [n] if setting else TREES:
            model.estimator.base.trees_ = k
            tried[(str(params), k)] = mae_log(model.predict(rows[f["valid"]]), f["valid"])
    key, trees = min(tried, key=tried.get)
    model = fitted[key]
    model.estimator.base.trees_ = trees
    params = model.estimator.base.params
    return {"model": model, "params": {**params, "trees": trees},
            "setting": {"params": params, "trees": trees},
            "tried": {f"{p} {t}": round(v, 5) for (p, t), v in tried.items()}}


for year, f in FOLDS.items():
    fit = fit_boost(f, year)
    run("lightgbm", year, fit)
    if year in SCORED:
        best = sorted(fit["tried"].items(), key=lambda kv: kv[1])[:3]
        print(year, "chosen:", fit["setting"], "| best three on the validation year:", best)
display(pd.DataFrame(RESULTS).query("level == 'lightgbm'").set_index("year"))

## 7. Comparison and choice

Each level's measures per test year. The deciding measure is the mean absolute error of the log
(V2-56). The choice (V2-50): of the levels whose gap to the best is within one standard error of
their four yearly gaps, the simplest. Price-only stocks and US filers are shown apart.

In [ ]:
# Comparison of the four levels and the year-by-year one-standard-error choice (V2-50)
ORDER = ["baseline", "rule", "ridge", "lightgbm"]  # simplest first
res = pd.DataFrame(RESULTS)
MEASURES = ["mae_log", "rmse_log", "rank_corr", "coverage_80", "mae_log_filers",
            "mae_log_price_only", "coverage_price_only"]
for measure in MEASURES:
    print(measure + ("  (deciding, V2-56)" if measure == "mae_log" else ""))
    display(res.pivot(index="level", columns="year", values=measure).loc[ORDER])

LL = res.pivot(index="year", columns="level", values="mae_log")[ORDER]
best = LL.mean().idxmin()
gap = LL.sub(LL[best], axis=0)  # how much worse than the best, per test year
summary = pd.DataFrame({
    "mean absolute log error": LL.mean(),
    "gap to the best (mean)": gap.mean(),
    "SE of the yearly gaps": gap.std() / np.sqrt(len(LL)),
    "years the best is better": (gap > 0).sum(),
}).loc[ORDER]
summary["as good as the best"] = (summary["gap to the best (mean)"]
                                  <= summary["SE of the yearly gaps"])
CHOSEN = next(level for level in ORDER if summary.loc[level, "as good as the best"])
display(summary)
print("gap to the best per test year:")
display(gap.T.loc[ORDER])
print(f"best mean absolute log error: {best}; CHOSEN LEVEL: {CHOSEN} (the simplest level whose "
      "gap to the best is within one standard error of its yearly gaps, V2-50)")
spread = LL[best].std() / np.sqrt(len(LL))
textbook = next(level for level in ORDER if LL[level].mean() <= LL[best].mean() + spread)
print(f"for reference, the textbook version (one standard error of the best level's own yearly "
      f"scores, {spread:.4f}) would choose: {textbook}")
CHECKS = pd.DataFrame({"beats the baseline": LL[CHOSEN] < LL["baseline"],
                       "beats the rule": LL[CHOSEN] < LL["rule"]})
display(CHECKS)
print("Note: the 2022 test year covers January-September only (later windows end after 2022).")

## 8. The level, the 80% range and the bands

**The range** (V2-10): for each test year, the 10th and 90th percentiles of the chosen level's
out-of-sample errors in all earlier years (from 2014) whose answers were known before the year.
The table shows the two multipliers (the range runs from the level times the first to the level
times the second) and how often the answer fell inside: 80% is honest.

**The bands**: five groups by the forecast level, with edges at the quintiles of the same earlier
years' forecasts, so a band means the same from year to year. For each band and test year: the
rows, the median forecast, the median real volatility and the range's coverage (D47).

**The chart**: rows in ten groups by their forecast; an honest level lies on the diagonal (forecast
0.40, real median 0.40).

In [ ]:
# The chosen level's 80% range, five bands and a calibration chart per test year
ranges = {}
for year in SCORED:
    t = TEST[(CHOSEN, year)]
    r = t["range"].estimator
    m = res.query("level == @CHOSEN and year == @year").iloc[0]
    ranges[year] = {"low multiplier": np.exp(-r.q90_), "high multiplier": np.exp(-r.q10_),
                    "history years": f"{t['history_years'][0]}-{t['history_years'][-1]}",
                    "coverage": m["coverage_80"], "coverage, price-only": m["coverage_price_only"]}
RANGES = pd.DataFrame(ranges).T
print(f"{CHOSEN}: the 80% range per test year (level x low multiplier to level x high multiplier)")
display(RANGES)

band_rows, rising = [], {}
for year in SCORED:
    f, t = FOLDS[year], TEST[(CHOSEN, year)]
    y = rows.loc[f["test"], "log_vol"].to_numpy()
    d = pd.DataFrame({"band": t["band"], "forecast": np.exp(t["forecast"]), "real": np.exp(y),
                      "inside": (y >= t["low"]) & (y <= t["high"])})
    g = d.groupby("band").agg(rows=("real", "size"), median_forecast=("forecast", "median"),
                              median_real=("real", "median"), coverage=("inside", "mean"))
    rising[year] = bool(g["median_real"].is_monotonic_increasing)
    band_rows.append(g.assign(year=year).reset_index())
BAND_TABLE = pd.concat(band_rows).set_index(["year", "band"])
edges = np.exp(TEST[(CHOSEN, SCORED[-1])]["bands"].estimator.edges_)
print(f"bands of {CHOSEN} per test year; the 2022 edges (annual volatility): {np.round(edges, 3)}")
display(BAND_TABLE)
print("median real volatility rises from band 1 to band 5:", rising)

fig, axes = plt.subplots(1, len(SCORED), figsize=(16, 4), sharey=True)
for ax, year in zip(axes, SCORED, strict=True):
    f, t = FOLDS[year], TEST[(CHOSEN, year)]
    d = pd.DataFrame({"f": np.exp(t["forecast"]),
                      "y": np.exp(rows.loc[f["test"], "log_vol"].to_numpy())})
    d["group"] = pd.qcut(d["f"].rank(method="first"), 10, labels=False)
    g = d.groupby("group").median()
    top = float(max(g["f"].max(), g["y"].max()))
    ax.plot([0, top], [0, top], color="grey", lw=1, ls="--")
    ax.plot(g["f"], g["y"], marker="o")
    ax.set_title(str(year))
    ax.set_xlabel("forecast (median of the group)")
axes[0].set_ylabel("real volatility (median)")
fig.suptitle(f"{CHOSEN}: forecast against real volatility, ten groups per test year")
plt.tight_layout()
plt.show()

## 9. What the model uses, and a leak check

The strongest inputs of the ridge regression and of LightGBM (2021 fold), and a leak check: no
input outside the feature list, and no single feature near-perfect on its own.

In [ ]:
# What the models use and a leak check (research R8)
fit = FITTED[("ridge", 2021)]["model"].estimator
pipe, cols = fit.base, fit.cols
coef, imputer = pipe[-1].coef_, pipe[2]
kept = [c for c, s in zip(cols, imputer.statistics_, strict=True) if np.isfinite(s)]
labels = kept + [f"{cols[i]} (was missing)" for i in imputer.indicator_.features_]
assert len(labels) == len(coef)
display(pd.Series(coef, index=labels).sort_values(key=np.abs, ascending=False).head(20)
        .rename("standardised coefficient (ridge on the change, 2021 fold)").to_frame())

boost = FITTED[("lightgbm", 2021)]["model"].estimator.base
gain = pd.Series(boost.model_.booster_.feature_importance("gain", iteration=boost.trees_),
                 index=MODEL_FEATURES)
display((gain / gain.sum()).sort_values(ascending=False).head(20)
        .rename("share of LightGBM gain (change, 2021 fold)").to_frame()
        .assign(family=lambda t: [FAMILY[c] for c in t.index]))

# Leak check: inputs are features only, and no single feature is near-perfect on its own
assert set(LINEAR) <= set(MODEL_FEATURES) and not NEVER & set(FEATURES)
assert not set(MACRO) & set(MODEL_FEATURES)  # V2-57
tr = FOLDS[2019]["train"]
single = rows.loc[tr, FEATURES].corrwith(rows.loc[tr, "log_vol"], method="spearman")
print("strongest single features (rank correlation with the log answer, 2019 fold's training "
      "rows):")
display(single.abs().sort_values(ascending=False).head(10).rename("|rank correlation|").to_frame())
assert (single.abs() < 0.95).all()

## 9b. The hypotheses (V2-46, docs/hypotheses.md)

Each hypothesis names a family of inputs and why it should matter for the next three months'
volatility. **Family check:** the chosen level (LightGBM if a rule level was chosen) is refitted
without the family on every fold (the macro family, left out since V2-57, is added back instead),
with that fold's settings, five seeds (one for the ridge); the
family **adds** if the mean absolute log error is worse without it, averaged over the four test
years, by more than one standard error of the four yearly differences. The change models keep
`log(vol_63)` as their starting point when the volatility family is removed. **Direction check:**
the rank correlation of the key feature with the answer in each test year; the expected sign in at
least three of the four. **Verdict:** supported (adds, right direction), not supported (adds
nothing), opposite (the direction is reversed in at least three years).

| # | Family | Why | Expected |
|---|---|---|---|
| H-V1 | past volatility | turbulence persists but drifts back toward normal | higher past, higher future; the rule's slope below 1 |
| H-V2 | size and trading | small, thinly traded companies swing more than their past suggests | smaller, higher |
| H-V3 | financials | fragile companies react more to any news | loss-making, higher |
| H-V4 | macro | market-wide fear moves every stock | higher market volatility, higher |
| H-V5 | insider, short volume | no clear mechanism | no effect |
| H-V6 | all, combined | the company's features improve on the one-line rule | the model beats the rule by more than one standard error |

In [ ]:
# The hypotheses for volatility (docs/hypotheses.md): family checks and direction checks
CHECK_LEVEL = CHOSEN if CHOSEN in ("ridge", "lightgbm") else "lightgbm"
BASE = MODEL_FEATURES if CHECK_LEVEL == "lightgbm" else LINEAR
SEEDS = range(5) if CHECK_LEVEL == "lightgbm" and not TRIAL else range(1)  # ridge has no seed
GROUP = {
    "H-V1 past volatility": VOL,
    "H-V2 size and trading": [c for c in FEATURES if FAMILY[c] in ("size", "trading")],
    "H-V3 financials": [c for c in FEATURES if FAMILY[c] == "financials"],
    "H-V4 macro": [c for c in FEATURES if FAMILY[c] == "macro"],
    "H-V5 insider and short volume": [c for c in FEATURES
                                      if FAMILY[c] in ("insider", "short volume")],
}
# (with the family, without it): removed for the families the models use, added back for macro
CHECKS = {h: (BASE, [c for c in BASE if c not in g]) for h, g in GROUP.items() if h != "H-V4 macro"}
CHECKS["H-V4 macro"] = (BASE + MACRO, BASE)  # left out since V2-57: checked by adding it back
VARIANTS = sorted({tuple(v) for pair in CHECKS.values() for v in pair})


def refit(level: str, year: int, cols, seed: int) -> dict:
    """The level refitted on `cols` with the fold's chosen setting and `seed`."""
    f, fitted = FOLDS[year], FITTED[(level, year)]
    if level == "lightgbm":
        return fit_boost(f, year, cols, seed, fitted["setting"])
    return fit_level(lambda alpha: ridge(alpha, cols), f, year, [fitted["params"]])


CHECK = {(cols, year): np.mean([mae_log(refit(CHECK_LEVEL, year, list(cols), s)["model"]
                                        .predict(rows[FOLDS[year]["test"]]),
                                        FOLDS[year]["test"]) for s in SEEDS])
         for cols in VARIANTS for year in SCORED}
family_rows = []
for h, (with_f, without_f) in CHECKS.items():
    d = pd.Series({year: CHECK[(tuple(without_f), year)] - CHECK[(tuple(with_f), year)]
                   for year in SCORED})  # positive: worse without the family
    se = d.std() / np.sqrt(len(d))
    family_rows.append({"check": h, "inputs": len(set(with_f) ^ set(without_f)),
                        **{str(year): v for year, v in d.items()}, "mean": d.mean(), "se": se,
                        "adds": bool(d.mean() > se)})
FAMILY_TABLE = pd.DataFrame(family_rows).set_index("check")
print(f"mean absolute log error without the family minus with it ({CHECK_LEVEL}, seeds "
      f"{list(SEEDS)}): positive means the family helps")
display(FAMILY_TABLE)

# Direction checks: rank correlation of each key feature with the answer, per test year
KEY = {"H-V1": ("vol_63", +1), "H-V2": ("log_market_cap", -1), "H-V3": ("loss_making", +1),
       "H-V4": ("spy_vol_63", +1)}
test_rows = rows[np.logical_or.reduce([FOLDS[y]["test"].to_numpy() for y in SCORED])]
rho = eda.rank_by_year(test_rows, [k for k, _ in KEY.values()], ANSWER)
display(rho)
slope_below_1 = bool((SLOPES.loc[SCORED] < 1).all())
print("H-V1: the rule's slope per scored fold", SLOPES.loc[SCORED].round(3).to_dict(),
      "| below 1 in every fold:", slope_below_1)

verdicts = []
for h, (feature, sign) in KEY.items():
    check = next(c for c in FAMILY_TABLE.index if c.startswith(h))
    right = int((np.sign(rho.loc[feature]) == sign).sum())
    wrong = int((np.sign(rho.loc[feature]) == -sign).sum())
    adds = bool(FAMILY_TABLE.loc[check, "adds"]) and (h != "H-V1" or slope_below_1)
    verdict = "opposite" if wrong >= 3 else ("supported" if adds and right >= 3
                                             else "not supported")
    verdicts.append({"hypothesis": check, "expected": f"{feature} {'+' if sign > 0 else '−'}",
                     "change without (mean ± SE)": f"{FAMILY_TABLE.loc[check, 'mean']:+.4f} ± "
                                                   f"{FAMILY_TABLE.loc[check, 'se']:.4f}",
                     "expected sign in years": f"{right} of {len(SCORED)}", "verdict": verdict})
h5 = FAMILY_TABLE.loc["H-V5 insider and short volume"]
verdicts.append({"hypothesis": "H-V5 insider and short volume", "expected": "no effect",
                 "change without (mean ± SE)": f"{h5['mean']:+.4f} ± {h5['se']:.4f}",
                 "expected sign in years": "–",
                 "verdict": "not supported" if h5["adds"] else "supported"})
model_level = CHOSEN if CHOSEN in ("ridge", "lightgbm") else "lightgbm"
lead = LL["rule"] - LL[model_level]  # positive: the model is better than the rule
lead_se = lead.std() / np.sqrt(len(lead))
verdicts.append({"hypothesis": "H-V6 all, combined", "expected": f"{model_level} beats the rule",
                 "change without (mean ± SE)": f"lead {lead.mean():+.4f} ± {lead_se:.4f}",
                 "expected sign in years": f"{int((lead > 0).sum())} of {len(SCORED)}",
                 "verdict": "supported" if lead.mean() > lead_se else "not supported"})
VERDICTS = pd.DataFrame(verdicts).set_index("hypothesis")
display(VERDICTS)
with mlflow.start_run(run_name="hypotheses"):
    mlflow.set_tags({**TAGS, "level": CHECK_LEVEL, "check": "hypotheses"})
    for check, r in FAMILY_TABLE.iterrows():
        mlflow.log_metric(re.sub(r"[^A-Za-z0-9_]", "_", check)[:60] + "_mean", r["mean"])

## 10. Frozen model and dry run

The chosen level made for the holdout exactly as for a test year (target year 2023): the forecast
learns from answers known before 2022 and takes its settings from the rows of 2022; its range and
band edges come from the out-of-sample history 2014–2022. One object of `Dated` steps with one
`predict` function (contract: `specs/005-volatility-model/contracts/model-contract.md`), fitted
twice to show the two fits agree. The dry run scores rows without answers: the 2023+ month-end
rows of `feature_snapshot.parquet` and the newest row of each company still trading
(`latest.parquet`). After the researcher approves, this code moves to
`src/sentinel/models/volatility.py` (FR-019).

In [ ]:
# The frozen model: the chosen level made for the holdout year exactly as for a test year
TARGET_YEAR = eda.DEV_END.year + 1
BAND_NAMES = dilution.BAND_NAMES
FIT = {"baseline": lambda f, y: fit_level(lambda: Dated(Baseline(), QUESTION, ["vol_63"]), f, y,
                                          [{}]),
       "rule": lambda f, y: fit_level(lambda: Dated(Rule(), QUESTION, ["vol_63"]), f, y, [{}]),
       "ridge": lambda f, y: fit_level(ridge, f, y, [{"alpha": a} for a in ALPHAS]),
       "lightgbm": lambda f, y: fit_boost(f, y)}


@dataclass
class VolatilityModel:
    """The model for one target year: the forecast, the 80% range and the band edges, each a
    `Dated` with its cutoff."""

    level: str
    features: list
    forecast: Dated
    range: Dated
    bands: Dated
    setting: dict
    seed: int
    year: int
    trained_through: str
    provenance: dict = field(default_factory=dict)
    eligibility_col: str = ELIG
    question: str = QUESTION


def fit_frozen(level: str, year: int = TARGET_YEAR, provenance=None) -> VolatilityModel:
    """`level` for target `year`: the forecast learns from answers known before `year` - 1 and takes
    its settings from the rows of `year` - 1; the range and the band edges come from the level's
    out-of-sample history before `year`."""
    f = eda.folds(rows, QUESTION, [year])[year]
    fit = FIT[level](f, year)
    hist = earlier(level, year)
    cutoff = pd.Timestamp(year, 1, 1)
    rng = Dated(Range(), QUESTION, ["forecast"]).fit(hist, cutoff, y=hist["error"])
    bands = Dated(dilution.Bands(), QUESTION, ["forecast"]).fit(hist, cutoff, y=hist["error"])
    return VolatilityModel(level, list(fit["model"].features), fit["model"], rng, bands,
                           fit["setting"], SEED, year,
                           str(rows.loc[f["inner"], "window_end"].max().date()), provenance or {})


def predict(model: VolatilityModel, rows_in: pd.DataFrame) -> pd.DataFrame:
    """One row per row of `rows_in` (same index): whether it is eligible and, for eligible rows, the
    expected volatility, its 80% range and the band (1-5) with its name. Every step refuses a row
    dated before its cutoff; a missing column is an error."""
    needed = ["cik", "as_of", model.eligibility_col, *model.features]
    missing = [c for c in needed if c not in rows_in.columns]
    if missing:
        raise KeyError(f"missing columns: {missing}")
    out = pd.DataFrame({
        "cik": rows_in["cik"].astype(str), "as_of": pd.to_datetime(rows_in["as_of"]),
        "eligible": rows_in[model.eligibility_col].fillna(False).astype(bool),
        "vol_forecast": np.nan, "vol_low_80": np.nan, "vol_high_80": np.nan,
        "vol_band": pd.Series(pd.NA, index=rows_in.index, dtype="Int64"),
        "vol_band_name": pd.Series(pd.NA, index=rows_in.index, dtype="string"),
    }, index=rows_in.index)
    mask = out["eligible"].to_numpy()
    if not mask.any():
        return out
    sub = rows_in.loc[mask].assign(as_of=out.loc[mask, "as_of"])
    sub = sub.astype({c: float for c in model.features})
    frame = sub[["as_of"]].assign(forecast=model.forecast.predict(sub))
    low, high = model.range.predict(frame).T
    band = model.bands.predict(frame)
    out.loc[mask, "vol_forecast"] = np.exp(frame["forecast"].to_numpy())
    out.loc[mask, "vol_low_80"] = np.exp(low)
    out.loc[mask, "vol_high_80"] = np.exp(high)
    out.loc[mask, "vol_band"] = band
    out.loc[mask, "vol_band_name"] = [BAND_NAMES[int(b)] for b in band]
    return out


made_from = {k: TAGS[k] for k in ("git_commit", "export_date", "data_hash")}
frozen = fit_frozen(CHOSEN, provenance=made_from)
again = fit_frozen(CHOSEN, provenance=made_from)
r = frozen.range.estimator
print(f"level {frozen.level}; cutoffs: forecast {frozen.forecast.cutoff.date()}, range "
      f"{frozen.range.cutoff.date()}, bands {frozen.bands.cutoff.date()}; learned from answers "
      f"known through {frozen.trained_through}; setting {frozen.setting}; range multipliers "
      f"{np.exp(-r.q90_):.3f} to {np.exp(-r.q10_):.3f}")

# Dry run on rows without answers (research R9)
snap = pq.read_table(eda.EXPORT / "feature_snapshot.parquet",
                     filters=[("as_of", ">", eda.DEV_END), ("month_end", "=", True)]).to_pandas()
latest = pd.read_parquet(eda.EXPORT / "latest.parquet")
latest["as_of"] = pd.to_datetime(latest["as_of"])
latest = latest[latest["as_of"] >= latest["as_of"].max() - pd.Timedelta(days=7)]  # still trading
COLS = ["vol_forecast", "vol_low_80", "vol_high_80"]
for name, frame in {"2023+ month ends": snap, "latest (still trading)": latest}.items():
    assert ANSWER not in frame.columns
    out, out2 = predict(frozen, frame), predict(again, frame)
    scored = out[out["eligible"]]
    eligible = int(frame[ELIG].fillna(False).astype(bool).sum())
    assert len(scored) == eligible and scored[COLS].notna().all().all()
    assert ((scored["vol_low_80"] < scored["vol_forecast"])
            & (scored["vol_forecast"] < scored["vol_high_80"])).all()
    assert scored["vol_band"].between(1, 5).all()
    assert out[COLS].equals(out2[COLS])  # two fits, identical forecasts
    bands = dict(sorted((int(b), int(n)) for b, n in scored["vol_band"].value_counts().items()))
    print(f"{name}: {len(frame):,} rows, {eligible:,} eligible, all scored, low < level < high, "
          f"two fits identical; median level {scored['vol_forecast'].median():.3f}; bands {bands}")

# The guard at work: the frozen model refuses a row dated before its cutoff
try:
    predict(frozen, rows[rows.as_of.dt.year == 2021].head(1))
    raise RuntimeError("a 2021 row was scored")
except AssertionError as err:
    print("refused as expected:", str(err)[:90])

with tempfile.TemporaryDirectory() as tmp, mlflow.start_run(run_name=f"frozen-{CHOSEN}"):
    path = Path(tmp) / "volatility_model.joblib"
    joblib.dump(frozen, path)
    mlflow.set_tags({**TAGS, "level": CHOSEN, "frozen": "candidate"})
    mlflow.log_params({"level": CHOSEN, "trained_through": frozen.trained_through,
                       "n_features": len(frozen.features),
                       "forecast_cutoff": str(frozen.forecast.cutoff.date())})
    mlflow.log_artifact(str(path))
print("frozen model saved in MLflow as a candidate (waiting for the researcher's approval)")

### Findings

*Export of 9 November 2026 (data hash `284e6f632544`); run at commit `0667dc2` in 20 minutes under
the memory cap (1,196 s), after V2-57. Every number is an aggregate from the section named in
brackets.*

#### In one paragraph

**LightGBM, forecasting the change from past volatility without the macro family, is the chosen
level** (V2-50). Its mean absolute log error is 0.279, against 0.290 for the ridge, 0.326 for the
mean-reversion rule and 0.346 for the naive baseline, and it is the best level in every test year.
In plain words, its typical miss is about 32% too high or too low, against about 41% for "past
volatility continues". Its 80% range held 78%, 72%, 84% and 88% of the answers in 2019–2022: close
to 80% in ordinary years, too narrow in the 2020 crash, a little wide in 2022. Five hypotheses
find what they expected; macro (H-V4) is the opposite. **One caveat, said plainly:** the macro
family was left out after the first full run (V2-57), that is after seeing the test years, so this
comparison flatters the models a little; the holdout test (010) is the honest check.

#### The first run and V2-57 [research R11]

The first full run (all 95 features, as planned) had LightGBM best in 2019–2021 (0.284, 0.381,
0.234) but worst in 2022 (0.332, against 0.254 for the baseline), so the year-by-year rule chose the
baseline. Without the macro family, LightGBM's 2022 error fell by 0.138: rates rose in 2022 faster
than in any training year. On 2016–2018, before every test year, LightGBM was also 0.003 better
without macro on average. The researcher left macro out of the ridge and LightGBM (V2-57), as for
dilution's trees (V2-36).

#### The levels [3–7]: mean absolute log error (deciding, V2-56; lower is better)

| Level | 2019 | 2020 | 2021 | 2022 | Mean |
|---|---|---|---|---|---|
| 0 Baseline (`vol_63` continues) | 0.3516 | 0.4901 | 0.2873 | 0.2540 | 0.3457 |
| 1 Rule (mean reversion, slope 0.73–0.78) | 0.3404 | 0.4296 | 0.2731 | 0.2608 | 0.3260 |
| 2 Ridge (the change, no macro) | 0.3053 | 0.3641 | 0.2523 | 0.2391 | 0.2902 |
| **3 LightGBM (the change, no macro)** | **0.2907** | **0.3591** | **0.2400** | **0.2264** | **0.2791** |

- Both models beat the baseline and the rule in every test year; LightGBM beats the ridge in every
  year (by 0.005 to 0.015). The same order holds for the root mean squared log error.
- **Ranking**: rank correlation with the answer 0.56–0.88 for LightGBM, 0.40–0.86 for the
  baseline; 2020 is hard for all (a calm end of 2019 said nothing about March 2020).
- **Price-only stocks** (about 1,000 rows a year): LightGBM's error 0.21–0.37, close to US
  filers' 0.23–0.36.
- The rule's slope is 0.73–0.78 in every fold: volatility moves about a quarter of the way back
  toward normal each quarter.

#### The choice [7]

- **Year-by-year rule (V2-50): LightGBM.** Its lead over the ridge is 0.011 on average with a
  standard error of 0.002 over the four yearly leads: it wins steadily.
- For reference, the textbook rule would keep the ridge (one standard error of LightGBM's own
  yearly scores is 0.030, larger than the gap).

#### The 80% range and the bands [8]

| | 2019 | 2020 | 2021 | 2022 |
|---|---|---|---|---|
| Range (level × low to level × high) | 0.69–1.50 | 0.69–1.48 | 0.70–1.60 | 0.70–1.57 |
| Share of answers inside (80% is honest) | 78% | 72% | 84% | 88% |
| Price-only stocks | 75% | 70% | 85% | 88% |

- The range is set on the out-of-sample errors of all earlier years from 2014 (V2-10). In ordinary
  years it is close to 80%; in the 2020 crash it was too narrow; once 2020's large errors joined
  its history, it ran a little wide in 2021–2022. Spec SC-005 (75–85% in three of four years) is met
  in two (2019, 2021): reported, not adjusted.
- **Bands**: band 1's stocks swung 17–19% a year (median) and band 5's 69–81% in 2019, 2021 and
  2022, rising from band to band. In 2020 the order broke at the bottom: the calmest stocks of
  late 2019 swung 52% a year in the crash (SC-006 met in three of four years).

#### What the model uses [9]

- Past volatility in its forms carries 72% of LightGBM's gain (`vol_63` 40%, `range_vol_21` 13%,
  `idio_vol_252` 9%, `vol_21` 8%); then the distance from the 12-month high, the time since listing,
  the price level, recent returns and a few financial ratios. The ridge agrees: the strongest
  coefficient is past volatility's, negative on the change (mean reversion).
- **Leak check passed**: no single feature reaches a rank correlation of 0.95 (strongest:
  `idio_vol_252`, 0.81).

#### The hypotheses [9b] (LightGBM, five seeds, every fold)

| Hypothesis | Change without the family | Expected sign | Verdict |
|---|---|---|---|
| H-V1 past volatility | +0.0506 ± 0.0146 | 4 of 4; the rule's slope below 1 | supported |
| H-V2 size and trading | +0.0006 ± 0.0002 | 4 of 4 | supported |
| H-V3 financials | +0.0028 ± 0.0017 | 4 of 4 | supported |
| H-V4 macro (added back) | −0.0493 ± 0.0486 | 1 of 4 | opposite |
| H-V5 insider and short volume | +0.0002 ± 0.0002 | – | supported (adds nothing) |
| H-V6 all, combined | lead over the rule +0.0469 ± 0.0087 | 4 of 4 | supported |

H-V2 and H-V3 add little but steadily. H-V4 is the opposite of the hypothesis: adding macro back
makes the forecasts worse (by 0.194 in 2022), and market volatility's direction is reversed in
three years.

#### The frozen model and the dry run [10]

- LightGBM with 63 leaves, at least 100 rows per leaf and 50 trees, chosen on the rows of 2022 (a
  year of new conditions, where fewer trees did best); it learned from answers known through
  30 December 2021; range multipliers 0.71 to 1.56 from the history 2014–2022 (cutoffs 1 Jan 2022
  and 1 Jan 2023). Fitted twice: identical (SC-002).
- **Dry run**: all 232,238 eligible 2023+ month-end rows and the newest rows of 5,030 companies
  still trading are scored, with the range around every level and a band; a 2021 row is refused
  (SC-007).

#### Open questions for the researcher

1. **Approve LightGBM** (the decided rule, V2-50), or keep the simpler ridge that the textbook rule
   would choose? Recommendation: LightGBM, the best level in every test year.
2. The range is too narrow in a crash and a little wide after one: reported as measured, for the
   model card (008); a range that widens with market conditions would be a new method (021).